# EDA — Dim Customers (UCI Online Retail II)

**Prerequisite:** `python scripts/uci_pipeline.py`  
**Source:** `data/modeling/uci_dim_customers.parquet`  
**Grain:** One row = one UCI customer (aggregates)

Pre-modeling checklist: overview · duplicates & NAs · correlation · target vs features · feature importance · collinearity pruning · train/val/test split (when n ≥ 500).


## Setup

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

SEED = 42
RANDOM_STATE = SEED
np.random.seed(SEED)
MIN_ROWS_3WAY = 500  # train / val / test when n >= this


def find_project_root() -> Path:
    path = Path.cwd().resolve()
    for candidate in (path, *path.parents):
        if (candidate / "scripts" / "build_datasets.py").exists():
            return candidate
    return path


PROJECT_ROOT = find_project_root()
DATA = PROJECT_ROOT / "data" / "modeling"
SPLITS = PROJECT_ROOT / "data" / "splits"
SPLITS.mkdir(parents=True, exist_ok=True)

PALETTE = {"primary": "#1f4e79", "accent": "#c0392b", "secondary": "#2ecc71", "neutral": "#7f8c8d"}

plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": "white", "axes.facecolor": "#fafafa",
    "axes.titleweight": "bold", "axes.titlesize": 13,
})


def load_parquet(name: str) -> pd.DataFrame:
    path = DATA / name
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}\nRun the prerequisite command from the notebook header.")
    return pd.read_parquet(path)


def audit_and_clean(
    df: pd.DataFrame,
    *,
    subset: list[str] | None = None,
    id_col: str | None = None,
    required_cols: list[str] | None = None,
    label: str = "dataset",
) -> pd.DataFrame:
    out = df.copy()
    n0 = len(out)
    dup_subset = subset if subset is not None else ([id_col] if id_col else None)
    n_dup = (
        out.duplicated(subset=dup_subset, keep="first").sum()
        if dup_subset else out.duplicated(keep="first").sum()
    )
    if n_dup:
        out = out.drop_duplicates(subset=dup_subset, keep="first")
    req = [c for c in (required_cols or []) if c in out.columns]
    na_rows = out[req].isna().any(axis=1).sum() if req else 0
    na_by_col = out[req].isna().sum() if req else pd.Series(dtype=int)
    if req:
        out = out.dropna(subset=req)
    print(f"[{label}] {n0:,} -> {len(out):,} rows | dup dropped: {n_dup:,} | NA rows dropped: {na_rows:,}")
    if na_rows and (na_by_col > 0).any():
        print("  NA by column:", na_by_col[na_by_col > 0].to_dict())
    return out


def plot_nulls(df: pd.DataFrame, title: str = "Missing values (%)"):
    null_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
    null_pct = null_pct[null_pct > 0]
    fig, ax = plt.subplots(figsize=(8, max(3, len(null_pct) * 0.35)))
    if null_pct.empty:
        ax.text(0.5, 0.5, "No missing values", ha="center", va="center", fontsize=12)
        ax.set_title(title)
    else:
        null_pct.plot(kind="barh", ax=ax, color=PALETTE["accent"])
        ax.set_xlabel("Null %")
        ax.set_title(title)
    plt.tight_layout()
    plt.show()
    return null_pct


def correlation_heatmap(frame: pd.DataFrame, title: str):
    if frame.shape[1] < 2:
        print("Need >= 2 numeric columns for correlation heatmap.")
        return
    corr = frame.corr()
    fig, ax = plt.subplots(figsize=(max(8, len(corr) * 0.45), max(6, len(corr) * 0.4)))
    sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax, annot=len(corr) <= 12, fmt=".2f")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
    return corr


def prune_collinear(corr: pd.DataFrame, features: list[str], threshold: float = 0.85) -> list[str]:
    present = [c for c in features if c in corr.columns]
    if len(present) < 2:
        return present
    sub = corr.loc[present, present]
    upper = sub.where(np.triu(np.ones(sub.shape), k=1).astype(bool))
    drop = {c for c in upper.columns if any(upper[c].abs() > threshold)}
    selected = [c for c in present if c not in drop]
    print(f"Collinearity prune (|r|>{threshold}): drop {sorted(drop) or 'none'}")
    print(f"Selected ({len(selected)}):", selected)
    return selected


def target_correlations(num_df: pd.DataFrame, target: str) -> pd.Series:
    if target not in num_df.columns:
        return pd.Series(dtype=float)
    return num_df.corr()[target].drop(target, errors="ignore").sort_values(key=abs, ascending=False)


def quick_feature_importance(X, y, task: str = "classification", top_n: int = 15):
    from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline

    est = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1) if task == "classification" else RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
    pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", est)])
    pipe.fit(X, y)
    imp = pd.Series(pipe.named_steps["model"].feature_importances_, index=X.columns).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(8, max(4, top_n * 0.3)))
    imp.head(top_n).iloc[::-1].plot(kind="barh", ax=ax, color=PALETTE["primary"])
    ax.set_title(f"Random Forest feature importance (top {top_n})")
    plt.tight_layout()
    plt.show()
    return imp


def split_train_val_test(
    df: pd.DataFrame,
    *,
    feature_cols: list[str],
    target_col: str,
    id_col: str | None = None,
    group_col: str | None = None,
    stratify_col: str | None = None,
    split_name: str = "split",
    task: str = "classification",
):
    """60/20/20 when n >= MIN_ROWS_3WAY; else 70/30 train/test."""
    from sklearn.model_selection import train_test_split

    work = df.dropna(subset=[c for c in feature_cols + [target_col] if c in df.columns]).copy()
    n = len(work)
    strat = work[stratify_col] if stratify_col and stratify_col in work.columns and task == "classification" else None

    if group_col and group_col in work.columns:
        groups = work[[group_col]].drop_duplicates()
        if strat is not None and stratify_col in work.columns:
            grp_label = work.groupby(group_col)[stratify_col].first()
            groups = groups.merge(grp_label.rename("_s"), left_on=group_col, right_index=True)
            strat_g = groups["_s"]
        else:
            strat_g = None
        if n >= MIN_ROWS_3WAY:
            g_train, g_temp = train_test_split(groups[group_col], test_size=0.40, random_state=RANDOM_STATE, stratify=strat_g)
            g_temp_df = groups[groups[group_col].isin(g_temp)]
            strat_temp = g_temp_df["_s"] if strat_g is not None and "_s" in g_temp_df.columns else None
            g_val, g_test = train_test_split(g_temp_df[group_col], test_size=0.50, random_state=RANDOM_STATE, stratify=strat_temp)
            train = work[work[group_col].isin(g_train)]
            val = work[work[group_col].isin(g_val)]
            test = work[work[group_col].isin(g_test)]
            scheme = "60/20/20 (grouped)"
        else:
            g_train, g_test = train_test_split(groups[group_col], test_size=0.30, random_state=RANDOM_STATE, stratify=strat_g)
            train = work[work[group_col].isin(g_train)]
            val = work.iloc[0:0]
            test = work[work[group_col].isin(g_test)]
            scheme = "70/30 train/test (grouped — too few rows for val)"
    else:
        X = work[feature_cols]
        y = work[target_col]
        if n >= MIN_ROWS_3WAY:
            X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.40, random_state=RANDOM_STATE, stratify=strat)
            X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp if strat is not None else None)
            train = work.loc[X_train.index]
            val = work.loc[X_val.index]
            test = work.loc[X_test.index]
            scheme = "60/20/20"
        else:
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=strat)
            train = work.loc[X_train.index]
            val = work.iloc[0:0]
            test = work.loc[X_test.index]
            scheme = "70/30 train/test (too few rows for val)"

    print(f"Split scheme: {scheme}")
    print(f"Train {len(train):,} · Val {len(val):,} · Test {len(test):,}")
    if task == "classification" and target_col in work.columns:
        for name, part in [("train", train), ("val", val), ("test", test)]:
            if len(part):
                print(f"  {name} {target_col} rate: {part[target_col].mean():.1%}")

    out_dir = SPLITS / split_name
    out_dir.mkdir(parents=True, exist_ok=True)
    cols = [c for c in [id_col, group_col, target_col, *feature_cols] if c and c in work.columns]
    train[cols].to_parquet(out_dir / "train.parquet", index=False)
    if len(val):
        val[cols].to_parquet(out_dir / "val.parquet", index=False)
    test[cols].to_parquet(out_dir / "test.parquet", index=False)
    print(f"Saved splits -> {out_dir.relative_to(PROJECT_ROOT)}")
    return train, val, test

print("EDA utilities loaded · data:", DATA)


## 1 · Load & overview

In [ ]:
PARQUET = "uci_dim_customers.parquet"
SPLIT_NAME = "uci_dim_customers"
ID_COL = 'customer_id'
TARGET = 'total_revenue'
TARGET_TYPE = 'regression'
GROUP_COL = None

df_raw = load_parquet(PARQUET)

print(f"Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} cols")
print("Columns:", list(df_raw.columns))
display(df_raw.head(3))
display(df_raw.describe(include="all").T.head(20))


## 2 · Data quality (duplicates & missing)

In [ ]:
req = ['customer_id', 'total_revenue']
df = audit_and_clean(
    df_raw,
    subset=['customer_id'],
    id_col='customer_id',
    required_cols=req,
    label="uci_dim_customers",
)
nulls = plot_nulls(df, "Missing values (%)")
dup_report = df_raw.duplicated(subset=['customer_id']).sum() if ['customer_id'] else df_raw.duplicated().sum()
print(f"Duplicate rows (raw, before clean): {dup_report:,}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")


In [ ]:
eda_df = df.copy()

## 3 · Target & univariate distributions

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
eda_df[TARGET].hist(bins=40, ax=ax, color=PALETTE["primary"])
ax.set_title(f"Target distribution: {TARGET}")
ax.axvline(eda_df[TARGET].median(), color=PALETTE["accent"], ls="--", label="median")
ax.legend()
plt.tight_layout()
plt.show()
print(eda_df[TARGET].describe().round(2))


## 4 · Correlation & target relationships

In [ ]:
FEATURE_COLS = [
    "total_orders", "total_lines", "avg_unit_price", "avg_order_value",
    "recency_days", "tenure_days", "points_balance", "app_usage_score", "discount_sensitivity",
]

FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]
print("Modeling features:", FEATURE_COLS)

num = eda_df[[c for c in FEATURE_COLS + [TARGET] if c in eda_df.columns]].select_dtypes(include=[np.number])
corr = correlation_heatmap(num, "Feature correlation matrix")

if TARGET in num.columns:
    tc = target_correlations(num, TARGET)
    fig, ax = plt.subplots(figsize=(7, max(4, min(12, len(tc)) * 0.35)))
    tc.head(15).iloc[::-1].plot(kind="barh", ax=ax, color=PALETTE["secondary"])
    ax.set_title(f"Top correlations with {TARGET}")
    plt.tight_layout()
    plt.show()
    display(tc.head(10).to_frame("corr_with_target"))


## 5 · Target vs feature plots

In [ ]:
plot_cols = [c for c in FEATURE_COLS if c in df.columns and pd.api.types.is_numeric_dtype(df[c])][:6]
if not plot_cols:
    print("No numeric features to plot.")
else:
    n = len(plot_cols)
    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    axes = axes.flatten()
    for i, col in enumerate(plot_cols):
        ax = axes[i]
        if TARGET_TYPE == "binary" and TARGET in df.columns:
            groups = df.groupby(TARGET)[col]
            ax.boxplot([groups.get_group(g).dropna() for g in sorted(df[TARGET].dropna().unique())], labels=sorted(df[TARGET].unique()))
            ax.set_title(f"{col} by {TARGET}")
        else:
            ax.scatter(df[col], df[TARGET], alpha=0.35, s=12, color=PALETTE["primary"])
            ax.set_xlabel(col)
            ax.set_ylabel(TARGET)
            ax.set_title(f"{TARGET} vs {col}")
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    plt.tight_layout()
    plt.show()


## 6 · Feature importance & selection

In [ ]:

model_df = df.copy()
X_imp = model_df[[c for c in FEATURE_COLS if c in model_df.columns]].select_dtypes(include=[np.number])
y_imp = model_df[TARGET]
if len(X_imp.columns) and len(model_df) >= 30:
    imp = quick_feature_importance(X_imp, y_imp, task="regression")
    display(imp.head(10).to_frame("importance"))
else:
    imp = None
    print("Skip importance — insufficient rows or features.")

if 'corr' in dir() and corr is not None:
    SELECTED = prune_collinear(corr, FEATURE_COLS)
else:
    SELECTED = FEATURE_COLS


## 7 · Train / validation / test split

In [ ]:
split_features = [c for c in (SELECTED if 'SELECTED' in dir() else FEATURE_COLS) if c in df.columns]
if len(split_features) and TARGET in df.columns and len(df) >= 50:
    train, val, test = split_train_val_test(
        df,
        feature_cols=split_features,
        target_col=TARGET,
        id_col=ID_COL,
        group_col=GROUP_COL,
        stratify_col='total_revenue',
        split_name=SPLIT_NAME,
        task="regression",
    )
else:
    print("Split skipped — need target, features, and >= 50 rows.")


## Summary

| Item | Value |
|------|-------|
| **Dataset** | `uci_dim_customers.parquet` |
| **Target** | `total_revenue` |
| **Split saved** | `data/splits/uci_dim_customers/` |

Proceed to the matching modeling notebook in `Notebooks/01`–`05` after reviewing signals above.
